In [ ]:
import pyodbc

conn = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=tu_servidor;"
    "DATABASE=CRONOX;"
    "UID=tu_usuario;"
    "PWD=tu_password;"
)

cursor = conn.cursor()

# Ejecutar el stored procedure
cursor.execute("EXEC dbo.SP_Generador_CENCOSUD_SAE")

# Confirmar cambios (MUY IMPORTANTE)
conn.commit()

cursor.close()
conn.close()

print("Stored Procedure ejecutado correctamente")

In [1]:
import sys 
sys.path.append('C:/Users/DATA/Documents/datos/01_script/inicio/funciones')
from funciones import *
from funciones_spark import *
from variables_inicio import *
from utils_sql import *


## inventario

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
# from pyspark.sql.functions import last_day, col
from dateutil.relativedelta import relativedelta

spark = SparkSession.builder \
    .appName("SparkExample") \
    .master("local[*]") \
    .config('spark.driver.extraClassPath', 'C:/spark/jars/mssql-jdbc-13.2.1.jre11.jar') \
    .config('spark.executor.extraClassPath', 'C:/spark/jars/mssql-jdbc-13.2.1.jre11.jar') \
    .config('spark.executor.memory', '8g') \
    .config('spark.driver.memory', '8g') \
    .getOrCreate()

fecha_mes_base = '2026-03-01'
descripcion_01 = "''IVR - NO CONFIRMA','IVR - SOLICITA CONTACTO CON EJECUTIVO','MALA EXPERIENCIA (TDA/FNC)','MAL SERVICIO','SOLICITA SER RETIRADO'"
tipo_01 = "'EFECTIVA','EFECTIVA - CONTACTO NO LOGRADO'"

query = f"""
    select 
    ROW_NUMBER() OVER (ORDER BY (SELECT NULL)) AS indice
    ,a.Dni as [VENDOR LEAD CODE]
    ,a.PHONE_NUMBER as [PHONE NUMBER]
    ,a.Fecha_Llamada as fecha_llamada
    from [SAMANTHA].[dbo].[tmp_llamadas_mes] a
    inner join [ODIN].[dbo].[tTipologia_Efectiva] b
    on a.Codigo_Paleta=b.CODIGO
    where RIGHT(a.Nombre_Campana, 8) = 'CTIVA_NC'
    and a.Fecha_Llamada BETWEEN DATEADD(MONTH, -6, '{fecha_mes_base}')
                        AND '{fecha_mes_base}'
    and b.TIPO in ({tipo_01})
    and a.Dni<> ''
    and b.DESCRIPCION not in ({descripcion_01})
    """
df_efe_seis_meses=obtener_tabla_sql(spark,query,server_zeus,user_zeus,pwd_zeus,db_zeus)

window_spec = Window.partitionBy("VENDOR LEAD CODE").orderBy(col("fecha_llamada").desc())
df_efe_seis_meses = df_efe_seis_meses.withColumn("ref_01", row_number().over(window_spec))
df_efe_seis_meses = df_efe_seis_meses.filter(col("ref_01") == 1).drop('ref_01')

query = """
    SELECT *
    FROM [CRONOX].[dbo].[Llave_Financiera_Efe] 
    WHERE Mejor_Descripcion_Telefono NOT IN ('NUMERO EQUIVOCADO','NO DESEA SER CONTACTADO / MANIFIESTA SU RECHAZO'
    ,'TITULAR NO CALIFICA - DESEMPLEO','SOLICITA SER RETIRADO','TITULAR NO CALIFICA- GIRO RESTRINGIDO9',
    'CLIENTE RECHAZADO','PERSONA FALLECIDA')
    and Mejor_Estado_Cliente in ('NO CARGADO','SIN CONTACTO')
    """
df_efe_consumo_spark=obtener_tabla_sql(spark,query
,server_sa,user_sa,pwd_sa,db_sa)

cols_ref=df_efe_consumo_spark.columns
df_efe_consumo_spark=df_efe_consumo_spark.join(df_efe_seis_meses,["VENDOR LEAD CODE",'PHONE NUMBER'],"inner")

df_efe_consumo_spark=df_efe_consumo_spark[cols_ref]

overwrite_table_SQL(spark,df_efe_consumo_spark,'temp_inventario_e_consumo',server_zeus,user_zeus,pwd_zeus,'SAMANTHA')

spark.stop()


In [ ]:
query = "SELECT * FROM SAMANTHA.dbo.temp_inventario_e_consumo"
df_efe_consumo=get_data_sql(query, params_zeus)
archivo_csv = os.path.join(ruta_archivo_generado, "df_efe_consumo.csv")
df_efe_consumo.to_csv(archivo_csv, index=False,sep=';')

# para generar listas

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
# from pyspark.sql.functions import last_day, col
from dateutil.relativedelta import relativedelta

spark = SparkSession.builder \
    .appName("SparkExample") \
    .master("local[*]") \
    .config('spark.driver.extraClassPath', 'C:/spark/jars/mssql-jdbc-13.2.1.jre11.jar') \
    .config('spark.executor.extraClassPath', 'C:/spark/jars/mssql-jdbc-13.2.1.jre11.jar') \
    .config('spark.executor.memory', '8g') \
    .config('spark.driver.memory', '8g') \
    .getOrCreate()

fecha_mes_base = '2026-03-01'

query = f"""
    SELECT *
    FROM OPENQUERY([192.168.3.7], '
        SELECT        
        rtrim(ltrim(d.vendor_lead_code)) AS dni,        
        a.campaign_id AS Numero_Campana,        
        e.campaign_name AS Nombre_Campana,        
        a.user AS DNI_Ejecutivo,        
        c.full_name AS Ejecutivo,        
        a.call_date AS Fecha_Hora_Llamada,        
        a.length_in_sec AS segundos,        
        DATE(a.call_date) AS Fecha_Llamada,        
        HOUR(a.call_date) AS Trama_Hora,        
        '''' '''' AS Estados,        
        '''' '''' AS Sub_estado,        
        b.status_name AS Descripcion,        
        f.list_description,        
        f.list_name,        
        a.PHONE_NUMBER,        
        d.alt_phone as Fecha_Agenda,        
        d.comments as Comentarios,        
        a.status AS CODIGO,        
        FROM_UNIXTIME(a.start_epoch) as Inicio,        
        FROM_UNIXTIME(a.end_epoch) as Fin        
        FROM asterisk.vicidial_log a         
        LEFT JOIN asterisk.vicidial_list d ON a.lead_id=d.lead_id        
        LEFT JOIN asterisk.vicidial_campaigns e ON a.campaign_id=e.campaign_id        
        LEFT JOIN asterisk.vicidial_lists f ON a.list_id=f.list_id        
        LEFT JOIN asterisk.vicidial_statuses b ON a.status=b.status        
        LEFT JOIN asterisk.vicidial_users c ON a.user=c.user        
        WHERE call_date >= curdate()  and e.campaign_name like "%EFECTIVA_NC%"
    ')

    """
df_tmp_llamadas_efe_consumo=obtener_tabla_sql(spark,query,server_sa,user_sa,pwd_sa,db_sa)

In [ ]:
df_tmp_llamadas_efe_consumo=df_tmp_llamadas_efe_consumo.join(df_efe_tipif_spark,["CODIGO"],"left")

window_spec = Window.partitionBy("dni","PHONE_NUMBER").orderBy(col("PESO").asc(),col("Fecha_Llamada").desc())
df_tmp_llamadas_efe_consumo = df_tmp_llamadas_efe_consumo.withColumn("mejor_Tel", row_number().over(window_spec))

window_spec = Window.partitionBy("dni").orderBy(col("PESO").asc(),col("Fecha_Llamada").desc())
df_tmp_llamadas_efe_consumo = df_tmp_llamadas_efe_consumo.withColumn("mejor_resul", row_number().over(window_spec))

window_spec = Window.partitionBy("dni",'PHONE_NUMBER').orderBy(col("Fecha_Hora_Llamada").desc())
df_tmp_llamadas_efe_consumo = df_tmp_llamadas_efe_consumo.withColumn("ult_llamada_tel", row_number().over(window_spec))

window_spec = Window.partitionBy("dni").orderBy(col("Fecha_Hora_Llamada").desc())
df_tmp_llamadas_efe_consumo = df_tmp_llamadas_efe_consumo.withColumn("ult_llamada_cli", row_number().over(window_spec))

window_count = Window.partitionBy("dni", "PHONE_NUMBER")
df_tmp_llamadas_efe_consumo = df_tmp_llamadas_efe_consumo.withColumn(
    "nCant_agent",
    count(
        when(col("DNI_Ejecutivo") != "VDAD", 1)
    ).over(window_count)
)
df_tmp_llamadas_efe_consumo = df_tmp_llamadas_efe_consumo.withColumn(
    "nCant_no_agent",
    count(
        when(col("DNI_Ejecutivo") == "VDAD", 1)
    ).over(window_count)
)
window_count = Window.partitionBy("dni")
df_tmp_llamadas_efe_consumo = df_tmp_llamadas_efe_consumo.withColumn("q_llamada_cli",count("*").over(window_count))
window_count = Window.partitionBy("dni", "PHONE_NUMBER")
df_tmp_llamadas_efe_consumo = df_tmp_llamadas_efe_consumo.withColumn("q_llamada_cli_tel",count("*").over(window_count))

# df_tmp_llamadas_efe_consumo = df_tmp_llamadas_efe_consumo.filter(col("mejor_Tel") == 1).drop('mejor_Tel')


In [9]:
query = """
    SELECT *
    FROM [ODIN].[dbo].[tTipologia_Efectiva]
    """
df_efe_tipif_spark=obtener_tabla_sql(spark,query,server_zeus,user_zeus,pwd_zeus,db_zeus)

df_efe_tipif_spark.show()

+------+--------+-----------------+---------------+--------------------+----------+----+
|CODIGO|COD_TIPO|             TIPO|SUB_DESCRIPCION|         DESCRIPCION|CODIGO_BCO|PESO|
+------+--------+-----------------+---------------+--------------------+----------+----+
|  B001|       1|CONTACTO EFECTIVO|       EFECTIVO|  CREDITO CONCRETADO|     AMK00|   1|
|  B002|       1|CONTACTO EFECTIVO|       EFECTIVO|           SI QUIERE|     AMK01|   2|
|  B003|       1|CONTACTO EFECTIVO|       EFECTIVO|SOLICITA SER RETI...|     DMK01|   3|
|  B004|       1|CONTACTO EFECTIVO|       EFECTIVO|      LO VA A PENSAR|     AMK02|   5|
|  B005|       1|CONTACTO EFECTIVO|       EFECTIVO|           NO QUIERE|     AMK03|   6|
|  B006|       1|CONTACTO EFECTIVO|       EFECTIVO|      ESTA ENDEUDADO|     AMK04|   7|
|  B007|       1|CONTACTO EFECTIVO|       EFECTIVO|NO NECESITA / NO ...|     AMK05|   8|
|  B008|       1|CONTACTO EFECTIVO|       EFECTIVO|     DESEA MAS MONTO|     AMK06|   9|
|  B009|       1|CONT

In [ ]:
['Dni', 'Numero_Campana', 'Nombre_Campana', 'DNI_Ejecutivo', 'Ejecutivo', 'Fecha_Hora_Llamada', 'segundos', 'Fecha_Llamada', 'Trama_Hora', 'Estados', 'Sub_estado', 'Descripcion', 'list_description', 'list_name', 'PHONE_NUMBER', 'Fecha_Agenda', 'Comentarios', 'Codigo_Paleta', 'Inicio', 'Fin']codi


['Dni', 'Numero_Campana', 'Nombre_Campana', 'DNI_Ejecutivo', 'Ejecutivo', 'Fecha_Hora_Llamada', 'segundos', 'Fecha_Llamada', 'Trama_Hora', 'Estados', 'Sub_estado', 'Descripcion', 'list_description', 'list_name', 'PHONE_NUMBER', 'Fecha_Agenda', 'Comentarios', 'Codigo_Paleta', 'Inicio', 'Fin']


In [12]:
df_tmp_llamadas_efe_consumo.show()

+--------+--------------+-------------------+-------------+--------------------+-------------------+--------+-------------+----------+-------+----------+--------------------+----------------+----------------+------------+------------+--------------------+-------------+-------------------+-------------------+
|     Dni|Numero_Campana|     Nombre_Campana|DNI_Ejecutivo|           Ejecutivo| Fecha_Hora_Llamada|segundos|Fecha_Llamada|Trama_Hora|Estados|Sub_estado|         Descripcion|list_description|       list_name|PHONE_NUMBER|Fecha_Agenda|         Comentarios|Codigo_Paleta|             Inicio|                Fin|
+--------+--------------+-------------------+-------------+--------------------+-------------------+--------+-------------+----------+-------+----------+--------------------+----------------+----------------+------------+------------+--------------------+-------------+-------------------+-------------------+
|22519831|           123|2026-03 EFECTIVA_NC|         VDAD|  Outbound 

TypeError: cannot unpack non-iterable StringDtype object

In [11]:
df_efe_consumo.count()

LAST NAME           20083
FIRST NAME          20083
ADDRESS1            20083
ADDRESS3              385
COMMENTS                0
VENDOR LEAD CODE    20083
PROVINCE            20083
CITY                20083
EMAIL               20046
COMMENTS_2          20083
PHONE NUMBER        20083
dtype: int64

In [13]:
df_efe_consumo.count()

9947

In [8]:
print(df_efe_consumo.columns)
print(df_efe_seis_meses.columns)

['LAST NAME', 'FIRST NAME', 'ADDRESS1', 'ADDRESS3', 'COMMENTS', 'VENDOR LEAD CODE', 'PROVINCE', 'CITY', 'EMAIL', 'COMMENTS_2', 'PHONE NUMBER', 'Order_Numero', 'Oferta', 'Tip_Prioridad', 'MARCA_2025', 'NOMCOMERCIAL', 'Perfil_ic', 'tipocliente', 'TIPOINGRESO', 'ASIGNACION', 'LINEA_FT', 'LINEA_HS_RS', 'LINEA_HS_PLUS', 'LINEA_FULL', 'MARCA', 'TIP_TELF', 'PROVEEDOR', 'PERFIL', 'SEGMENTO', 'SCORE', 'TASA', 'RANGO_TASA', 'ZONA', 'Tipo_Telf', 'Fecha_Llam', 'Hora_Llamada', 'FECHA_ENVIO', 'RETIRO', 'Mejor_Estado_Telefono', 'Mejor_Sub_Estado_Telefono', 'Mejor_Descripcion_Telefono', 'Mejor_Fecha_Telefono', 'Mejor_Hora_Telefono', 'Mejor_Durac_Telefono', 'Mejor_Estado_Cliente', 'Mejor_Sub_Estado_Cliente', 'Mejor_Descripcion_Cliente', 'Mejor_fecha_Cliente', 'Mejor_Hora_Cliente', 'Mejor_Durac_Cliente', 'ULT_Descripcion_Cli', 'ULT_fecha_Cli', 'ULT_Hora_Cli', 'ULT_Durac_Cli', 'ULT_Descripcion_Tel', 'ULT_fecha_Tel', 'ULT_Hora_Tel', 'ULT_Durac_Tel', 'Veces_Cliente', 'Veces_Telefono', 'REPITENCIAS', 'Canti

In [ ]:
df_efe_consumo

window_spec = Window.partitionBy("Dni").orderBy(col("dni").desc(),col("fecha_llamada").desc())
df_02 = df_02.withColumn("ref_01", row_number().over(window_spec))
df_02 = df_02.filter(col("ref_01") == 1).drop('ref_01')

In [9]:
df_efe_consumo.show()

+--------+------------+-------------+-------------------+-------------+--------------------+---------------+
|     Dni|PHONE_NUMBER|Codigo_Paleta|     Nombre_Campana|Fecha_Llamada|         DESCRIPCION|SUB_DESCRIPCION|
+--------+------------+-------------+-------------------+-------------+--------------------+---------------+
|40459213|   933080269|         B005|2026-01 EFECTIVA_NC|   2026-01-02|           NO QUIERE|       EFECTIVO|
|29599389|   977801128|         B005|2026-01 EFECTIVA_NC|   2026-01-02|           NO QUIERE|       EFECTIVO|
|46996439|   967129079|         B016|2026-01 EFECTIVA_NC|   2026-01-02|     VOLVER A LLAMAR|       EFECTIVO|
|26955889|   964718482|         B005|2026-01 EFECTIVA_NC|   2026-01-02|           NO QUIERE|       EFECTIVO|
|77477396|   961796589|         B004|2026-01 EFECTIVA_NC|   2026-01-02|      LO VA A PENSAR|       EFECTIVO|
|17548615|   979900524|         B006|2026-01 EFECTIVA_NC|   2026-01-02|      ESTA ENDEUDADO|       EFECTIVO|
|80102309|   982779

In [7]:

query = """
select
    DESCRIPCION
,codigo
,SUB_DESCRIPCION
from [ODIN].[dbo].[tTipologia_Efectiva] 
where TIPO in ('CONTACTO EFECTIVO')
and DESCRIPCION in ('IVR - NO CONFIRMA','IVR - SOLICITA CONTACTO CON EJECUTIVO',
'MALA EXPERIENCIA (TDA/FNC)','MAL SERVICIO','SOLICITA SER RETIRADO')
"""
df_efe_tipi=obtener_tabla_sql(spark,db_zeus,query,server_zeus,user_zeus,pwd_zeus)
df_efe_tipi.show()

+--------------------+------+---------------+
|         DESCRIPCION|codigo|SUB_DESCRIPCION|
+--------------------+------+---------------+
|SOLICITA SER RETI...|  B003|       EFECTIVO|
|MALA EXPERIENCIA ...|  B010|       EFECTIVO|
|        MAL SERVICIO|  B011|       EFECTIVO|
+--------------------+------+---------------+



In [4]:
df_efe_consumo.show()

+---+------------+-------------+--------------+-------------+-----------+---------------+
|Dni|PHONE_NUMBER|Codigo_Paleta|Nombre_Campana|Fecha_Llamada|DESCRIPCION|SUB_DESCRIPCION|
+---+------------+-------------+--------------+-------------+-----------+---------------+
+---+------------+-------------+--------------+-------------+-----------+---------------+



In [8]:
df_efe_consumo['FIRST NAME'].unique()

<StringArray>
['POTENCIALES', 'DORMIDO', 'AFILIADOS', 'NUEVO']
Length: 4, dtype: str

In [6]:
df_efe_consumo.hist(column='FIRST NAME', bins=50)

ValueError: hist method requires numerical or datetime columns, nothing to plot.

In [65]:
orden_01=df_efe_consumo.columns.to_list

In [57]:
df_uno=pd.read_excel("C:\\Users\\DATA\\Documents\\datos\\01_script\\inicio\\27-02 SGT.xlsx")

In [58]:
df_uno.columns = (
    df_uno.columns
    .str.strip()
    .str.upper()
    .str.replace("Á","A")
    .str.replace("É","E")
    .str.replace("Í","I")
    .str.replace("Ó","O")
    .str.replace("Ú","U")
)

In [59]:
df_uno['PHONE NUMBER'] = df_uno['NUMERO'].astype(str)


In [60]:
df_uno=df_uno[['DNI', 'NOMBRE', 'PHONE NUMBER']]

In [61]:
df_uno=df_uno.merge(df_efe_consumo, left_on='PHONE NUMBER', right_on='PHONE NUMBER', how='left')

In [ ]:
Index(['LAST NAME', 'FIRST NAME', 'ADDRESS1', 'ADDRESS3', 'COMMENTS',
       'VENDOR LEAD CODE', 'PROVINCE', 'CITY', 'EMAIL', 'COMMENTS_2',
       'PHONE NUMBER', 'Order_Numero', 'Oferta', 'Tip_Prioridad', 'MARCA_2025',
       'NOMCOMERCIAL', 'Perfil_ic', 'tipocliente', 'TIPOINGRESO', 'ASIGNACION',
       'LINEA_FT', 'LINEA_HS_RS', 'LINEA_HS_PLUS', 'LINEA_FULL', 'MARCA',
       'TIP_TELF', 'PROVEEDOR', 'PERFIL', 'SEGMENTO', 'SCORE', 'TASA',
       'RANGO_TASA', 'ZONA', 'Tipo_Telf', 'Fecha_Llam', 'Hora_Llamada',
       'FECHA_ENVIO', 'RETIRO', 'Mejor_Estado_Telefono',
       'Mejor_Sub_Estado_Telefono', 'Mejor_Descripcion_Telefono',
       'Mejor_Fecha_Telefono', 'Mejor_Hora_Telefono', 'Mejor_Durac_Telefono',
       'Mejor_Estado_Cliente', 'Mejor_Sub_Estado_Cliente',
       'Mejor_Descripcion_Cliente', 'Mejor_fecha_Cliente',
       'Mejor_Hora_Cliente', 'Mejor_Durac_Cliente', 'ULT_Descripcion_Cli',
       'ULT_fecha_Cli', 'ULT_Hora_Cli', 'ULT_Durac_Cli', 'ULT_Descripcion_Tel',
       'ULT_fecha_Tel', 'ULT_Hora_Tel', 'ULT_Durac_Tel', 'Veces_Cliente',
       'Veces_Telefono', 'REPITENCIAS', 'Cantidad_Sin_Discador',
       'Cantidad_Discador', 'TELEFO1', 'TELEFO2', 'TELEFO3', 'REGION',
       'MARCADESBASE', 'MARCA2', 'FLGSUBPROCESO_HS', 'SITUACIONLABORAL']pho

Index(['LAST NAME', 'FIRST NAME', 'ADDRESS1', 'ADDRESS3', 'COMMENTS',
       'VENDOR LEAD CODE', 'PROVINCE', 'CITY', 'EMAIL', 'COMMENTS_2',
       'PHONE NUMBER', 'Order_Numero', 'Oferta', 'Tip_Prioridad', 'MARCA_2025',
       'NOMCOMERCIAL', 'Perfil_ic', 'tipocliente', 'TIPOINGRESO', 'ASIGNACION',
       'LINEA_FT', 'LINEA_HS_RS', 'LINEA_HS_PLUS', 'LINEA_FULL', 'MARCA',
       'TIP_TELF', 'PROVEEDOR', 'PERFIL', 'SEGMENTO', 'SCORE', 'TASA',
       'RANGO_TASA', 'ZONA', 'Tipo_Telf', 'Fecha_Llam', 'Hora_Llamada',
       'FECHA_ENVIO', 'RETIRO', 'Mejor_Estado_Telefono',
       'Mejor_Sub_Estado_Telefono', 'Mejor_Descripcion_Telefono',
       'Mejor_Fecha_Telefono', 'Mejor_Hora_Telefono', 'Mejor_Durac_Telefono',
       'Mejor_Estado_Cliente', 'Mejor_Sub_Estado_Cliente',
       'Mejor_Descripcion_Cliente', 'Mejor_fecha_Cliente',
       'Mejor_Hora_Cliente', 'Mejor_Durac_Cliente', 'ULT_Descripcion_Cli',
       'ULT_fecha_Cli', 'ULT_Hora_Cli', 'ULT_Durac_Cli', 'ULT_Descripcion_Tel',
       'U

In [68]:
df_uno=df_uno[['LAST NAME', 'FIRST NAME', 'ADDRESS1', 'ADDRESS3', 'COMMENTS',
       'VENDOR LEAD CODE', 'PROVINCE', 'CITY', 'EMAIL', 'COMMENTS_2',
       'PHONE NUMBER']]

In [ ]:
df_uno

In [69]:
df_uno.to_csv("C:\\Users\\DATA\\Documents\\datos\\01_script\\inicio\\base_01.csv", index=False,sep=';')

(593162, 71)

In [ ]:
# df_efe_consumo['Mejor_Sub_Estado_Cliente'].unique()
    ['Mejor_Descripcion_Cliente'].unique()

<StringArray>
[                             'NO QUIERE',
                 'NO NECESITA / NO DESEA',
                              'SI QUIERE',
                         'ESTA ENDEUDADO',
                        'DESEA MAS MONTO',
                        'VOLVER A LLAMAR',
                                  'OTROS',
                     'CREDITO CONCRETADO',
                         'LO VA A PENSAR',
                       'CLIENTE DE VIAJE',
                     'INTERESES ELEVADOS',
        'TITULAR NO CALIFICA - DESEMPLEO',
                        'ABONO EN CUENTA',
            'CASA EN ZONA NO COBERTURADA',
                  'SOLICITA SER RETIRADO',
         'NEGOCIO EN ZONA NO COBERTURADA',
             'MALA EXPERIENCIA (TDA/FNC)',
 'TITULAR NO CALIFICA- GIRO RESTRINGIDO9',
                         'CONFIRMAR CITA',
      'TITULAR CONTAGIADO CON COVID - 19',
                          'DESEA ELECTRO',
                      'CLIENTE RECHAZADO']
Length: 22, dtype: str

In [54]:
df_efe_consumo[df_efe_consumo['VENDOR LEAD CODE']=='47905540'].head()

,LAST NAME,FIRST NAME,ADDRESS1,ADDRESS3,COMMENTS,VENDOR LEAD CODE,PROVINCE,CITY,EMAIL,COMMENTS_2,...,Cantidad_Sin_Discador,Cantidad_Discador,TELEFO1,TELEFO2,TELEFO3,REGION,MARCADESBASE,MARCA2,FLGSUBPROCESO_HS,SITUACIONLABORAL


In [10]:
df_efe_consumo.columns

Index(['LAST NAME', 'FIRST NAME', 'ADDRESS1', 'ADDRESS3', 'COMMENTS',
       'VENDOR LEAD CODE', 'PROVINCE', 'CITY', 'EMAIL', 'COMMENTS_2',
       'PHONE NUMBER', 'Order_Numero', 'Oferta', 'Tip_Prioridad', 'MARCA_2025',
       'NOMCOMERCIAL', 'Perfil_ic', 'tipocliente', 'TIPOINGRESO', 'ASIGNACION',
       'LINEA_FT', 'LINEA_HS_RS', 'LINEA_HS_PLUS', 'LINEA_FULL', 'MARCA',
       'TIP_TELF', 'PROVEEDOR', 'PERFIL', 'SEGMENTO', 'SCORE', 'TASA',
       'RANGO_TASA', 'ZONA', 'Tipo_Telf', 'Fecha_Llam', 'Hora_Llamada',
       'FECHA_ENVIO', 'RETIRO', 'Mejor_Estado_Telefono',
       'Mejor_Sub_Estado_Telefono', 'Mejor_Descripcion_Telefono',
       'Mejor_Fecha_Telefono', 'Mejor_Hora_Telefono', 'Mejor_Durac_Telefono',
       'Mejor_Estado_Cliente', 'Mejor_Sub_Estado_Cliente',
       'Mejor_Descripcion_Cliente', 'Mejor_fecha_Cliente',
       'Mejor_Hora_Cliente', 'Mejor_Durac_Cliente', 'ULT_Descripcion_Cli',
       'ULT_fecha_Cli', 'ULT_Hora_Cli', 'ULT_Durac_Cli', 'ULT_Descripcion_Tel',
       'U